# Zajęcia: Wykorzystanie SpatioTemporal Asset Catalog (STAC)

## 1. Wprowadzenie do STAC

- **Co to jest STAC?**  
  - SpatioTemporal Asset Catalog 
  - Metadatane w formacie JSON: kolekcje, zasoby, itemy
  - Opisuje, w lekkiej postaci JSON/GeoJSON, zasoby geoprzestrzenne powiązane z konkretnym miejscem i czasem.  
- **Zalety:**
  - interoperacyjność, łatwe wyszukiwanie, filtrowanie
  - w praktyce wyszukujesz setki terabajtów danych satelitarnych jednym zapytaniem /search
  - link do dokumentacji STAC w ArcGIS Pro:  
  https://pro.arcgis.com/en/pro-app/latest/help/data/imagery/introduction-to-stac.htm


Od 2025 r. coraz więcej ekosystemów – m.in. Copernicus Data Space – publikuje katalogi zgodne z modelem STAC, integrując je z OGC API i językiem zapytań CQL2. Na razie STAC pozostaje najlżejszą – i najszerzej przyjętą – drogą do wyszukiwania danych satelitarnych.

## 2. Ręczne przeglądanie STAC w ArcGIS Pro

### 2.1. Utworzenie połączenia do STAC 

1. **ArcGIS Pro** → **Catalog** → **Add STAC Connection**  
2. Wprowadź URL usługi STAC (np. publiczny endpoint, lub wybierz Microsoft Planetary Computer)  
3. Nazwa połączenia (wpisz dowolną) **OK**  
4. Zajrzyj do nowego węzła w panelu Catalog  



### 2.2. Eksploracja kolekcji i itemów 
**Mini-zadanie 1:**  
- Dodaj połączenie do przykładowego endpointu **https://earth-search.aws.element84.com/v1**
- Sprawdź, czy widać listę kolekcji
- Wybierz dowolną kolekcję, np. Sentilnel-2
  
- **Filtrowanie**:  
  - po dacie (Date Range)  
  - po obszarze (Spatial Filter)  
  - po chmurach (Cloud Cover)  

Zastosuj dowolne filtry. Do szybkiego filtru Extent możesz użyć mapy projektu i wybrać opcję **Current Display Extent**.

- Wczytaj jeden item na mapę i zweryfikuj dane
- Przejrzyj metadane 

### 2.2. Wykorzystanie danych
**Mini-zadanie 2:**  
- Wykorzystaj istniejące połączenie do endpointu **https://earth-search.aws.element84.com/v1**
- Wybierz kolekcję *sentinel-2-l2a* i wybierz dwa zasoby (assets) - B04 i B08
  
 **Pozyskanie danych**:  
  - Wyszukaj zasoby dla dowolnego miejsca w Polsce, z pokryciem chmur poniżej 20%
  - Wczytaj jeden item na mapę i zweryfikuj dane
  - Wczytany zasób powinien mieć dwa kanały
    
 **Obliczenie NDVI**:
  - korzystając z kalkulatora rastrów oblicz NDVI z pozyskanych danych: (B08-B04)/(B08+B04)

## 3. Wprowadzenie do pracy z STAC w Pythonie 

Przejście od GUI do kodu.

Dokumentacja i strony warte odwiedzenia:

- https://github.com/Esri/arcgis-for-mpc
- https://github.com/stac-api-extensions/collection-search?rsource=https%3A%2F%2Flinks.esri.com%2FSTAC_API-CollectionSearch
- https://doc.arcgis.com/en/microsoft-planetary-computer/latest/help/access-data-on-microsoft-planetary-computer-through-arcpy.htm

#### 3.1. Import bibliotek i konfiguracja środowiska

W tej komórce importujemy wszystkie niezbędne moduły:

- **AIO** (z `arcpy`) – klasa do pracy z ArcGIS Image Analyst Object, umożliwia wczytanie pliku ACS (Authentication Configuration Store).
- **arcpy.ia** – moduł Image Analyst, udostępnia m.in. `RasterCollection`, funkcje rasterowe i obsługę złożonych analiz.

In [ ]:
import arcpy
import os
#import shutil
#import copy
#import random

from arcpy import AIO
from arcpy.ia import *

#### 3.2. Ładowanie pliku ACS (Authentication Configuration Store)

Plik ACS (*.acs*) zawiera parametry uwierzytelniające do usługi STAC:
- **Ścieżka** powinna wskazywać lokalizację pliku pobranego z repozytorium Esri.
- Wywołanie `a = AIO(ścieżka_do_pliku)` tworzy instancję klienta Image Analyst, która automatycznie:
  - ładuje ustawienia serwera,
  - przechowuje dane uwierzytelniające,
  - przygotowuje środowisko do dalszej pracy z kolekcjami STAC.

Odpowiednip plik należy pobrać stąd: https://github.com/Esri/arcgis-for-mpc/tree/874a88e21663709df3bb3e24f0b328381a485031/AMPC_Resources/ACS_Files

Wybieramy esrims_pc_landsat-c2-l2.acs

In [ ]:
a = AIO(r"C:\agh\SG\esrims_pc_landsat-c2-l2.acs")

#### 3.3. Definicja zapytania STAC i tworzenie RasterCollection

1. **Parametry zapytania** (`query`):
   - `collections`: nazwa kolekcji w STAC (tu: `"landsat-c2-l2"`).
   - `bbox`: obszar zainteresowania w układzie geograficznym [xmin, ymin, xmax, ymax].
   - `query`: dodatkowe warunki, np. platforma `"landsat-9"`.
   - `datetime`: zakres dat wykonywania ujęć w formacie `YYYY-MM-DD/YYYY-MM-DD`.
   - `limit`: maksymalna liczba zwracanych scen.

2. **Tworzenie RasterCollection**:
   - attribute_dict mapuje nazwy atrybutów STAC na klucze słownika Pythona.

Rezultatem jest obiekt rc, zawierający listę elementów (itemów), z których każdy ma np. klucze "Raster", "Name", "StdTime", itp.

In [ ]:
query = {
    "collections": ["landsat-c2-l2"],
    "bbox": [19.7, 49.8, 20.3, 50.2],
    "query": {"platform": {"in": ["landsat-9"]}, "eo:cloud_cover": {"lt": 20}},
    "datetime": "2025-01-01/2025-12-31",
    "limit": 3
          }
rc = arcpy.ia.RasterCollection.fromSTACAPI(stac_api="https://planetarycomputer.microsoft.com/api/stac/v1",
                                  query=query,
                                  attribute_dict={
                                      "Name":"id",
                                      "Cloud Cover":"eo:cloud_cover",
                                      "StdTime":"datetime",
                                      "Platform":"platform",
                                      "Spatial Reference":"proj:epsg",
                                      "Extent": "bbox"
                                    })

## 3.4. Podgląd struktury kolekcji RasterCollection

Sprawdzamy, co faktycznie pobraliśmy:
- Wydrukujemy sam obiekt `rc`, żeby zobaczyć liczbę elementów i ich atrybuty.
- `rc[0]['Raster']`, `rc[1]['Raster']`, `rc[2]['Raster']` – podgląd pierwszych trzech rastrów w kolekcji.
  
Dzięki temu upewnimy się, że kolekcja zawiera poprawne dane i będziemy mogli dalej je analizować.


In [ ]:
rc

In [ ]:
rc[0]['Raster']

In [ ]:
rc[1]['Raster']

In [ ]:
rc[2]['Raster']

#### 3.5. Wyciąganie pierwszego rastra i inspekcja dostępnych pasm

1. **Pobieramy pierwszy element** z kolekcji `rc` — to słownik zawierający m.in. obiekt rastra.  
2. Z tego słownika wyciągamy **rdzeniowy obiekt** `arcpy.Raster`.  
3. Opakowujemy go w klasę **Image Analyst Raster** (`arcpy.ia.Raster`), by uzyskać dostęp do metod charakterystycznych dla analiz obrazowych (np. `bandNames`).  
4. Wyświetlamy listę dostępnych **kanałów (pas­m)** w rastrze, żeby wiedzieć, które nazwy użyć dalej (np. do `selectBands` lub `getRasterBands`).

Dzięki temu upewniamy się, że znamy dokładne nazwy i kolejność pasm w scenie, co jest kluczowe przy obliczaniu NDVI.

In [ ]:

first_item = next(iter(rc))

core_ras   = first_item["Raster"]     # arcpy.Raster
ia_ras     = Raster(core_ras)         # teraz Image Analyst Raster

# sprawdzamy jakie mamy kanały
print("Dostępne kanały:", ia_ras.bandNames)

#### 3.6 Wybór pasm i definicja funkcji obliczającej NDVI

1. **Filtrowanie pasm**  
   - Metoda `selectBands()` na obiekcie `rc` zachowuje tylko pasmo czerwone (SRB4) i bliskiej podczerwieni (SRB5).  
   - Powstaje nowa kolekcja `rc2`, w której każdy raster ma dokładnie dwa pasma:  
     - `band1` → SRB4 (czerwone)  
     - `band2` → SRB5 (NIR)  

2. **Definicja funkcji `calc_ndvi`**  
   - Funkcja przyjmuje element kolekcji (`item`), gdzie `item["Raster"]` to obiekt rastra z dwoma pasmami.  
   - Oblicza wartość NDVI za pomocą operatora `(NIR – Red)/(NIR + Red)`, wskazując identyfikatory pasm:  
     - `nir_band_id=2` (pasmo NIR)  
     - `red_band_id=1` (pasmo czerwone)  
   - Zwraca słownik z przetworzonym rasterem NDVI oraz kluczowymi atrybutami (`Name`, `StdTime`).


In [ ]:
# Filtrowanie kolekcji do pasm czerwonego i NIR
rc2 = rc.selectBands(["SRB4", "SRB5"])

# Definicja funkcji obliczającej NDVI dla pojedynczego elementu
def calc_ndvi(item):
    ras2 = item["Raster"]         # obiekt rastra z dwoma pasmami: SRB4 i SRB5
   
    # Pasmo 1 = SRB4 (czerwone); Pasmo 2 = SRB5 (bliska podczerwień)
   
    ndvi_raster = NDVI(ras2, nir_band_id=2, red_band_id=1)
    return {
        "raster":  ndvi_raster,
        "Name":    item["Name"],
        "StdTime": item["StdTime"],
        # przekazujemy dalej inne potrzebne atrybuty...
    }


#### 3.7 Zastosowanie obliczeń NDVI na całej kolekcji

Teraz korzystamy z metody `map` obiektu `rc2` (RasterCollection), aby:

1. Dla każdego elementu kolekcji (`item`) wywołać funkcję `calc_ndvi(item)`,  
2. Wewnątrz `calc_ndvi` obliczyć wartości NDVI na podstawie wcześniej wybranych pasm Red i NIR,  
3. Zwrócić nowy słownik zawierający przetworzony raster NDVI oraz zachować oryginalne atrybuty (`Name`, `StdTime` itp.),  
4. Utworzyć nową kolekcję `ndvi_rc`, w której każdy element odpowiada scenie z kolekcji wejściowej, ale zamiast oryginalnego rastra zawiera obliczony NDVI.

W efekcie `ndvi_rc` to RasterCollection z rastrami NDVI gotowymi do dalszej analizy lub eksportu.


In [ ]:
# Zastosuj funkcję do całej kolekcji
ndvi_rc = rc2.map(calc_ndvi)

In [ ]:
ndvi_rc[0]['Raster']

In [ ]:
ndvi_rc[1]['Raster']

In [ ]:
ndvi_rc[2]['Raster']

# Projekt zaliczeniowy 

Celem projektu jest przygotowanie interaktywnego "podręcznika" („How-To”) w formie Jupyter Notebooka, w którym krok po kroku przeprowadzicie użytkownika przez pełen proces pobierania, przetwarzania i wizualizacji wybranej analizy na danych satelitarnych dostarczanych przez Microsoft Planetary Computer.

Przykładowe tematy 
- Zmiany wegetacji w czasie
- Analiza pokrycia chmur (policzyć % zachmurzenia i np. polizcyć histogram dla wybranego obszaru)
- Monitorowanie wód powierzchniowych (zmiana powierzchni wód) 
- Wykrywanie pożarów lub obszarów wypalonych
- Monitoring pokrywy śnieżnej
- Dowolny inny mający sens według grupy

# Wymagania projektowe i punktacja (20 pkt)

1. **Parametryzacja notebooka** (2 pkt)  
   - Sekcja na początku z definiowanymi zmiennymi:
     - `bbox = [xmin, ymin, xmax, ymax]`  
     - `date_range = "YYYY-MM-DD/YYYY-MM-DD"`  
     - opcjonalne filtry (jeśli potrzebne w analizie)  
   - Kod automatycznie odwołuje się do tych zmiennych, bez ręcznej edycji w dalszej części.

2. **Opis wybranej kolekcji** (2 pkt)  
   - Nazwa i charakterystyka kolekcji (sensor, rozdzielczość, typ produktu).  
   - Link do katalogu Planetary Computer.  
   - Uzasadnienie wyboru tej kolekcji dla wybranego scenariusza.

3. **Scenariusz analizy** (2 pkt)  
   - Jasne sformułowanie celu / pytania badawczego.  
   - Krótki opis metody np.:  
     - wzór na wskaźnik (np. NDVI, mNDWI, NBR)  
     - opis progowania / maskowania.  

4. **Implementacja krok po kroku** (6 pkt)  
   - Każdy fragment kodu poprzedzony komórką Markdown z wyjaśnieniem (po polsku lub angielsku, można utrzymać dowolny język dla całego notebooka).  
   - Przykładowe kolejne etapy:  
        - Importy i autoryzacja (AIO, licencje).
        - Tworzenie obiektu RasterCollection z STACAPI.
        - Eksploracja metadanych i atrybutów.
        - Filtrowanie scen 
        - Definicja funkcji obliczeniowej 
        - Zastosowanie funkcji na całej kolekcji (map).
        - Dalsze obliczenia (np. różnica między scenami, analizowanie trendów itp.)
     
5. **Wizualizacja wyników** (3 pkt)  
   - Wyświetlenie wynikowych rastrów (należy zadbać w miarę możliwości o symbolizację).  
   - Dodatkowe wykresy (np. histogram, wykres czasowy wartości wskaźnika dla wybranego piksela/obszaru)  
        
6. **Podsumowanie i uwagi** (2 pkt)  
   - Krótka interpretacja uzyskanych wyników.  
   - Ograniczenia wybranej metody i propozycje poprawek.  
   - Inne końcowe uwagi dla potencjalnego użytkownika.

7. **Kompleksowość i samodzielność** (1 pkt)  
   - Notebook musi być kompletny, czytelny i w pełni samodzielny.  
   - Jeśli wymagane pliki zewnętrzne (np. `.acs`), należy opisać, skąd je pobrać i jak ustawić ścieżki.  
   - Wszystkie ścieżki do zapisu wyników umieść w osobnej, wyróżnionej sekcji.

8.  **Prezentacja** (2 pkt)
    - na zajęciach należy przedstawić Notebook reszcie grupy i wyjaśnić co się w nim znajduje, jakie były trudności i jaki jest efekt pracy.  

> **Uwaga:** celem nie jest sztywne trzymanie się przedstawionych kroków, ale przygotowanie interaktywnego i czytelnego notebooka, który w ciekawy sposób przeprowadzi użytkownika przez przykład wykorzystania danych STAC z Microsoft Planetary Computer.


> **Uwaga:** projekt można wykonać samodzielnie lub w grupach dwu, trzy-osobowych 